# 🎯 [QLT04_M05] 정비 자원 재배치 시 예상 가동률 상승폭 시뮬레이션 산출 엔진 개발 (훈련생 실습용)
### 훈련생 실습 워크북 (Google Colab / Jupyter Student Edition)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/QLT04/qlt04_m05_student.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 품질·규격·검사 (Level 2) |
| **미션 ID** | QLT04_M05 |
| **미션 주제** | 정비 자원 재배치 시 예상 가동률 상승폭 시뮬레이션 산출 엔진 개발 |
| **소요 시간** | 약 3시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
외주정비는 정비 대기, 외주 검사 대기, 재작업, 규격·제원 불일치가 겹치면서 장비 가동률을 떨어뜨리지만 요인별 영향이 한눈에 보이지 않아 어디에 정비 자원을 먼저 투입할지 정하기 어렵다. 본 프로젝트는 미 국방부 외주정비 품질보증 사례의 문제의식(규격 정합성·검사 병목·하자 연결)을 교육용 가상 자료로 재구성해, 정비 대기 요인 분류, 저가동 요인 분석, 재배치 시뮬레이션, 추세 대시보드, 사람 검토 워크플로우까지 프로토타입으로 구현하는 과제이다. 가상 장비는 유형 A·B·C와 가상 부대로만 구분하며 실제 장비·부대 자료는 쓰지 않는다. 최종 목표는 가동률 저하 요인과 외주 품질검사 병목을 근거와 함께 보여 주되, 정비 우선순위와 자원 투입 결정은 정비관·품질검사관·군수참모가 하도록 지원하는 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

> **💡 [오프라인 폐쇄망 및 로컬 주피터 환경 안내]**
> - 외부 인터넷망(Colab) 외에 군 인트라넷 또는 로컬 환경에서 실행할 경우, 아래 명령어로 필요한 표준 라이브러리를 사전 설치하십시오:
>   `pip install numpy pandas matplotlib scipy scikit-learn`
> - 인터넷이 제한된 환경의 경우 사전 번들링된 Wheel(`.whl`) 패키지 디렉토리를 통해 설치할 수 있습니다:
>   `pip install --no-index --find-links=./wheels/ -r requirements.txt`

> 📁 **[안내: 프로젝트 표준 산출물 디렉터리 가이드]**
> - 본 프로젝트는 단순한 일회성 코드 실습을 넘어 실제 부대에 배포 가능한 업무 도구(Tool)를 완성하는 과정입니다.
> - 실습 중 작성하는 결과물은 역할에 따라 아래 표준 디렉터리에 자동으로 저장·누적됩니다:
>   - `docs/`: 프로세스 정의서(`qlt04_as_is_to_be_spec.md`), UAT 피드백 리포트(`qlt04_uat_feedback_report.md`), 운영 매뉴얼(`qlt04_final_manual.md`) 등 행정·기술 문서
>   - `src/`: 데이터 로더, 16종 품질 판정 룰 엔진, 릿지 회귀 XAI 엔진, 시뮬레이터 UI, HITL 결재 FSM, 배치 러너 등 핵심 실행 모듈
>   - `dist/`: 폐쇄망 및 야전 배포를 위한 최종 패키지(`qlt04_package/`)

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 05. 정비 자원 재배치 시 예상 가동률 상승폭 시뮬레이션 산출 엔진 개발
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

정비 자원 재배치 시 예상 가동률 상승폭 시뮬레이션 산출 엔진을 개발한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **정비병과 부품 재고를 부대 사이에서 옮겼을 때 예상 가동률 상승폭을 시뮬레이션하는 엔진을 만들고, 입력-분석-검토-결과 4개 화면의 프로토타입으로 보여 준다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **재고·충원율 부족분의 제곱과 정비 요청 건수의 상호작용을 반영한 회귀 모델을 만들어 검증 R2 확인하고 유효성을 점검하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/ui/qlt04_app.py` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 군 실무자 친화적 디자인 및 조작 편의성

#### 미션 PASS 조건

- 정량 기준: 핵심 화면 4종(입력-분석-검토-결과) 구현 완료


- 진행 기준: UI 프로토타입 로컬 서버 구동 및 전체 페이지 네비게이션 확인


- 검증 방법: 사용성 평가(UX Walkthrough) 및 인터랙션 테스트


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""부대별 정비병 충원율·부품 재고 가용률을 다른 부대로 옮겼을 때 장비 가동률이 얼마나 오르는지 회귀 모델로 시뮬레이션하고 상승폭이 큰 재배치안을 순서대로 고르며, 입력-분석-검토-결과 4개 화면과 상태 뱃지(승인대기·반려·완료)·필터·화면 이동을 갖춘 텍스트 프로토타입으로 보여 준다."""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 장비 가동률·정비 대기 자료 생성기 (실제 부대·장비·제원·가동률 정보 없음)
# 장비 유형 A(궤도형)·B(화포형)·C(차륜형)은 추상 구분이며 실제 장비와 무관하다.
# 최근 90일 동안 장비 1대의 정비 대기 요인(부품·정비병·외주 정비·외주 검사·재작업)과 가동률을 만든다.
# ---------------------------------------------------------------------------
UNITS = ["가상A부대", "가상B부대", "가상C부대", "가상D부대", "가상E부대", "가상F부대"]
ETYPES = ["유형A", "유형B", "유형C"]
WINDOW = 90                                       # 관측 기간(일)
CAUSES = ["부품 대기", "정비병 부족", "외주 정비", "검사·재작업"]
CAUSE_COL = {"부품 대기": "parts_wait_days", "정비병 부족": "labor_wait_days", "외주 정비": "outsource_days", "검사·재작업": "inspect_wait_days"}
NUM_FIELDS = ["age_years", "op_hours", "maint_orders", "parts_wait_days", "labor_wait_days", "outsource_days", "inspect_wait_days", "rework_count",
              "spec_mismatch", "defect_reports", "parts_stock_pct", "mechanic_fill_pct", "avail_pct"]
FEAT = ["age_years", "maint_orders", "parts_stock_pct", "mechanic_fill_pct", "outsource_days", "inspect_wait_days", "rework_count", "spec_mismatch", "defect_reports", "etype_B", "etype_C"]
FEAT_KO = {"age_years": "장비 연령", "maint_orders": "정비 요청 건수", "parts_stock_pct": "부품 재고 가용률", "mechanic_fill_pct": "정비병 충원율", "outsource_days": "외주 정비 일수",
           "inspect_wait_days": "외주 검사 대기 일수", "rework_count": "재작업 건수", "spec_mismatch": "규격·제원 불일치", "defect_reports": "하자·불만 접수",
           "etype_B": "유형B 여부", "etype_C": "유형C 여부"}


def unit_profile(seed=3):
    """부대별 고정 특성: 부품 재고 수준, 정비병 충원율, 외주 정비소 거리(일수 가중), 외주 검사 혼잡도."""
    r = np.random.default_rng(seed)
    return pd.DataFrame({"unit": UNITS, "stock": r.uniform(45, 95, len(UNITS)), "fill": r.uniform(55, 98, len(UNITS)),
                         "dist": r.uniform(0.8, 1.6, len(UNITS)), "queue": r.uniform(0.7, 2.2, len(UNITS))})


def make_fleet(n=300, seed=7, shock=0.0):
    """장비 n대의 90일 관측 자료(필드 16개)와 정답 요인을 만든다. shock은 외주 검사 병목 악화 정도(AAR용 환경 변화)."""
    rng = np.random.default_rng(seed)
    pf = unit_profile()
    ui = rng.integers(0, len(UNITS), n)
    et = rng.choice(ETYPES, n, p=[.35, .3, .35])
    age = np.clip(rng.gamma(4, 2.2, n), 0.5, 30).round(1)
    eff = np.array([{"유형A": 1.15, "유형B": 1.0, "유형C": 0.9}[t] for t in et])
    orders = rng.poisson(np.clip(1.2 + 0.12 * age, 0.5, 6) * eff)
    stock = np.clip(pf["stock"].to_numpy()[ui] + rng.normal(0, 8, n), 20, 100).round(0)
    fill = np.clip(pf["fill"].to_numpy()[ui] + rng.normal(0, 6, n), 30, 100).round(0)
    parts_w = (orders * rng.gamma(2, 0.55 * (1 + ((100 - stock) / 28) ** 2) / 2)).round(0)
    labor_w = (orders * rng.gamma(2, 0.7 * (1 + ((100 - fill) / 24) ** 2) / 2)).round(0)
    n_out = rng.binomial(orders, 0.3)
    out_d = (n_out * rng.gamma(3, 4.5 * pf["dist"].to_numpy()[ui] / 3)).round(0)
    insp_w = (n_out * rng.gamma(2, (2.6 + shock) * pf["queue"].to_numpy()[ui] / 2)).round(0)
    spec = rng.poisson(0.2 + 0.04 * age)
    rework = rng.binomial(n_out, np.clip(0.08 + 0.1 * spec, 0, .6))
    defect = rng.poisson(0.3 + 0.7 * rework)
    base_rep = orders * 1.8
    down = base_rep + parts_w + labor_w + out_d + insp_w + rework * 4 + rng.normal(0, 1.2, n)
    comp = np.column_stack([parts_w, labor_w, out_d, insp_w + rework * 4])
    df = pd.DataFrame({"equip_id": [f"E-{i + 1:04d}" for i in range(n)], "unit": np.array(UNITS)[ui], "etype": et, "age_years": age,
                       "op_hours": np.clip(rng.normal(320 - 4 * age, 70, n), 40, 600).round(0), "maint_orders": orders,
                       "parts_wait_days": parts_w, "labor_wait_days": labor_w, "outsource_days": out_d, "inspect_wait_days": insp_w,
                       "rework_count": rework, "spec_mismatch": spec, "defect_reports": defect, "parts_stock_pct": stock, "mechanic_fill_pct": fill,
                       "avail_pct": (100 * (1 - np.clip(down, 0, WINDOW) / WINDOW)).round(1)})
    df["main_cause"] = np.where(comp.max(axis=1) < 3, "없음", np.array(CAUSES)[comp.argmax(axis=1)])
    return df


def feat_frame(df):
    """회귀·분석용 설명 변수 표(유형은 더미 변수)."""
    x = df.copy()
    x["etype_B"], x["etype_C"] = (x["etype"] == "유형B").astype(int), (x["etype"] == "유형C").astype(int)
    return x[FEAT]


def make_trend(months=12, seed=5, shock_unit=None):
    """[교육용가상자료] 부대 x 유형 x 월 평균 가동률 추세. shock_unit 부대는 하반기에 외주 검사 병목으로 가동률이 떨어진다."""
    rng = np.random.default_rng(seed)
    pf = unit_profile()
    rows = []
    for _, u in pf.iterrows():
        base = 62 + 0.22 * u["stock"] + 0.12 * u["fill"] - 3.0 * u["queue"]
        for t in ETYPES:
            off = {"유형A": -3.0, "유형B": 0.0, "유형C": 2.0}[t]
            for m in range(months):
                v = base + off + 2.0 * np.sin(m / 12 * 2 * np.pi) + rng.normal(0, 1.0)
                if shock_unit == u["unit"] and m >= months // 2:
                    v -= 1.5 * (m - months // 2 + 1)
                rows.append((u["unit"], t, m + 1, round(float(np.clip(v, 30, 99)), 1)))
    return pd.DataFrame(rows, columns=["unit", "etype", "month", "avail_pct"])

# ---- 가동률 회귀 모델과 저가동 요인 분류·추천 모델(외부 AI 서비스 없이 scikit-learn 사용) ----
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

CAUSE_COLS = ["age_years", "maint_orders", "parts_stock_pct", "mechanic_fill_pct", "outsource_days", "inspect_wait_days", "rework_count", "spec_mismatch",
              "defect_reports", "parts_wait_days", "labor_wait_days"]
ACTION = {"부품 대기": "부품 사전 청구·재고 보강", "정비병 부족": "정비병 재배치·교육 일정 조정", "외주 정비": "외주 정비 일정 조율·내부 정비 확대", "검사·재작업": "검사 일정 조율·규격 사전 대조"}


def cause_frame(df):
    x = df[CAUSE_COLS].copy()
    x["out_insp"] = x["outsource_days"] + x["inspect_wait_days"] + 4 * x["rework_count"]
    x["wait_pm"] = (x["parts_wait_days"] + x["labor_wait_days"]) / (x["maint_orders"] + 1)
    return x


def fit_avail(df, alpha=1.0):
    """가동률(%) 회귀: 표준화 + 릿지 회귀."""
    return make_pipeline(StandardScaler(), Ridge(alpha=alpha)).fit(feat_frame(df), df["avail_pct"])


def fit_cause(df, cut=85, C=5):
    """가동률이 cut% 미만인 장비의 주요 저가동 요인(4종)을 분류한다."""
    low = df[(df["avail_pct"] < cut) & (df["main_cause"] != "없음")]
    return make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=3000)).fit(cause_frame(low), low["main_cause"])


SIM_FEAT = ["age_years", "maint_orders", "spec_mismatch", "stock_gap_x_orders", "fill_gap_x_orders", "parts_stock_pct", "mechanic_fill_pct", "etype_B", "etype_C"]


def sim_frame(df):
    """자원 재배치 시뮬레이션용 설명 변수: 부족분이 클수록 대기가 더 빠르게 늘고 정비 요청 건수와 곱해져 커진다는 구조를 제곱·상호작용 항으로 둔다."""
    x = df.copy()
    x["etype_B"], x["etype_C"] = (x["etype"] == "유형B").astype(int), (x["etype"] == "유형C").astype(int)
    x["stock_gap_x_orders"] = x["maint_orders"] * ((100 - x["parts_stock_pct"]) / 28) ** 2
    x["fill_gap_x_orders"] = x["maint_orders"] * ((100 - x["mechanic_fill_pct"]) / 24) ** 2
    return x[SIM_FEAT]


def fit_sim(df, alpha=0.5):
    return make_pipeline(StandardScaler(), Ridge(alpha=alpha)).fit(sim_frame(df), df["avail_pct"])

fleet = make_fleet(600, seed=31)
sim_tr, sim_te = fleet.iloc[:420], fleet.iloc[420:]
sm = fit_sim(sim_tr)
print(f"[시뮬레이션 모델] 검증 R2 {r2_score(sim_te['avail_pct'], sm.predict(sim_frame(sim_te))):.3f} (재고·충원율 부족분의 제곱과 정비 요청 건수의 상호작용 반영)")
sm = fit_sim(fleet)
AUTH = 40                                                              # 부대당 정비병 편성 정원(가상 가정)
PARAM = {"max_move": 4, "min_donor_fill": 60.0, "stock_step": 8.0, "top_n": 4, "max_donor_drop": 3.5}


def unit_avail(df, unit, fill_d=0.0, stock_d=0.0):
    d = df[df["unit"] == unit].copy()
    d["mechanic_fill_pct"] = np.clip(d["mechanic_fill_pct"] + fill_d, 0, 100)
    d["parts_stock_pct"] = np.clip(d["parts_stock_pct"] + stock_d, 0, 100)
    return float(np.clip(sm.predict(sim_frame(d)), 0, 100).mean()), len(d)


def evaluate(df, kind, src, dst, n):
    """
    [TODO : 학생 실습 5] src 부대에서 dst 부대로 정비 자원(정비병 또는 부품)을 n단위 재배치할 때의 가동률 변화를 시뮬레이션하세요.
    - 힌트 1: 자원 종류(kind)에 따라 delta 변화량을 계산합니다:
      * 정비병: n / AUTH * 100
      * 부품재고: n * PARAM['stock_step'] / 2
    - 힌트 2: unit_avail 함수를 사용해 src 부대(감소: -delta)와 dst 부대(증가: +delta)의 가동률 변화(a1 - a0)를 계산합니다.
    - 힌트 3: 전체 평균 가동률 상승폭 gain = (a1 + na1 - a0 - na) / len(UNITS)를 계산합니다.
    - 힌트 4: 제약 조건(gain >= PARAM['gain_min'] 및 n <= PARAM['max_move'][kind]) 충족 시 '승인대기', 미충족 시 '반려' 상태 및 사유를 생성합니다.
    - 반환값: 제안 정보 딕셔너리 {'id', 'kind', 'src', 'dst', 'n', 'gain', 'src_delta', 'status', 'why'}
    """
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.


class App:
    SCREENS = ["입력", "분석", "검토", "결과"]

    def __init__(self):
        self.cur, self.flt, self.visited = "입력", None, ["입력"]

    def goto(self, s):
        assert s in self.SCREENS
        self.cur = s; self.visited.append(s); return self.render()

    def render(self):
        if self.cur == "입력":
            return ["입력 화면: 장비 %d대 / 부대 %d곳 / 파라미터 %s" % (len(fleet), len(UNITS), PARAM)]
        if self.cur == "분석":
            return [f"  {r.unit} 가동률 {r.avail:5.1f}% 재고 {r.stock:4.0f}% 충원 {r.fill:4.0f}% ({int(r.n)}대)" + (" <- 하위" if r.avail <= base["avail"].nsmallest(2).max() else "") for r in base.itertuples()]
        if self.cur == "검토":
            ps = [p for p in props if self.flt in (None, p["status"])]
            return [f"  {p['id']} {BADGE[p['status']]:8s} {p['kind']} {p['n']}단위 {p['src'][2:4]}->{p['dst'][2:4]} 전체 {p['gain']:+.2f}%p(수혜 {p['recv_gain']:+.1f}/제공 {-p['donor_drop']:+.1f}) | {p['note']}" for p in ps]
        ok = [p for p in props if p["status"] == "승인대기"]
        return [f"  승인 대기 {len(ok)}건 시행 시 예상 전체 가동률 +{sum(p['gain'] for p in ok):.2f}%p (정비병 이동은 합산 근사)", "  최종 시행은 군수참모 결정(미션7 워크플로우)"]


app = App()
for s in App.SCREENS:
    print(f"\n[화면 {App.SCREENS.index(s) + 1}: {s}]")
    for line in app.goto(s):
        print(line)
app.flt = "승인대기"; n1 = len(app.goto("검토")); app.flt = "반려"; n2 = len(app.goto("검토")); app.flt = None
print(f"\n필터 시험: 승인대기 {n1}건, 반려 {n2}건, 전체 {len(app.goto('검토'))}건")
nav = [(a, b) for a in App.SCREENS for b in App.SCREENS if a != b]
okn = 0
for a, b in nav:
    app.cur = a; okn += bool(app.goto(b))
print(f"화면 이동 시험 {okn}/{len(nav)} 성공, 핵심 화면 {len(App.SCREENS)}종 구현 (목표 4종: {'달성' if len(App.SCREENS) == 4 else '미달'})")
tot = sum(p["gain"] for p in props if p["status"] == "승인대기")
print(f"재배치 시뮬레이션: 승인대기 안 합산 예상 상승폭 +{tot:.2f}%p, 최대 단일 안 {max(c['gain'] for c in cands):+.2f}%p. 시뮬레이션은 가정 모델 기반 추정이며 시행 여부는 사람이 결정한다.")

plt.figure(figsize=(7, 4))
plt.bar([f"{p['id']}\n{ {'완료': 'done', '승인대기': 'pending', '반려': 'rejected'}[p['status']]}" for p in props], [p["gain"] for p in props], color=["tab:gray" if p["status"] == "완료" else "tab:blue" if p["status"] == "승인대기" else "tab:red" for p in props])
plt.ylabel("fleet availability gain (%p)"); plt.title("Reallocation proposals (gray done / blue pending / red rejected)"); plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 05 예상 실행 결과 (Output)</b></summary>

```text
[시뮬레이션 모델] 검증 R2 0.646 (재고·충원율 부족분의 제곱과 정비 요청 건수의 상호작용 반영)
후보 70건 평가 (제공 부대당 정비병 누적 최대 4명 이동, 제공 부대 충원율·재고 60% 이상 유지)

[화면 1: 입력]
입력 화면: 장비 600대 / 부대 6곳 / 파라미터 {'max_move': 4, 'min_donor_fill': 60.0, 'stock_step': 8.0, 'top_n': 4, 'max_donor_drop': 3.5}

[화면 2: 분석]
  가상A부대 가동률  78.3% 재고   50% 충원   76% (94대)
  가상B부대 가동률  75.0% 재고   57% 충원   63% (87대) <- 하위
  가상C부대 가동률  83.8% 재고   84% 충원   87% (99대)
  가상D부대 가동률  77.8% 재고   75% 충원   59% (115대) <- 하위
  가상E부대 가동률  80.8% 재고   50% 충원   72% (100대)
  가상F부대 가동률  79.9% 재고   66% 충원   78% (105대)

[화면 3: 검토]
  P00 [완료]     정비병 2단위 C부->D부 전체 +0.14%p(수혜 +1.5/제공 -0.9) | 전 분기 시행 완료(가상 이력)
  P01 [반려]     정비병 4단위 C부->B부 전체 +0.11%p(수혜 +3.1/제공 -2.0) | 제공 부대 누적 이동 6명(한도 4명 초과)
  P02 [승인대기]   부품 재고 4단위 C부->A부 전체 +0.06%p(수혜 +0.4/제공 -0.1) | 제공 부대 영향·누적 이동 허용 범위
  P03 [승인대기]   정비병 2단위 C부->E부 전체 +0.05%p(수혜 +1.2/제공 -0.9) | 제공 부대 영향·누적 이동 허용 범위
  P04 [반려]     정비병 2단위 C부->F부 전체 +0.04%p(수혜 +1.1/제공 -0.9) | 제공 부대 누적 이동 6명(한도 4명 초과)

[화면 4: 결과]
  승인 대기 2건 시행 시 예상 전체 가동률 +0.10%p (정비병 이동은 합산 근사)
  최종 시행은 군수참모 결정(미션7 워크플로우)

필터 시험: 승인대기 2건, 반려 2건, 전체 5건
화면 이동 시험 12/12 성공, 핵심 화면 4종 구현 (목표 4종: 달성)
재배치 시뮬레이션: 승인대기 안 합산 예상 상승폭 +0.10%p, 최대 단일 안 +0.26%p. 시뮬레이션은 가정 모델 기반 추정이며 시행 여부는 사람이 결정한다.
```
</details>
